#### Load the pickle file

In [3]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

#### Load the trained model, scaler pickle and onehot pickle

In [4]:
model=load_model('my_model.keras')

##### Load the encoder and scaler

In [5]:
with open('onehot_encoder_geo.pkl','rb') as file:
    one_geo_encoder=pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_gender=pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)

#### Example Input Data

In [6]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

#### One-hot Encode 'Geopgraphy'

In [8]:
geo_encoded = one_geo_encoder.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=one_geo_encoder.get_feature_names_out(['Geography']))
geo_encoded_df

C:\Users\ayush\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [18]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


#### Encode Categorical Variables

In [19]:
input_df['Gender']=label_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


#### Concatenation

In [20]:
input_df=pd.concat([input_df.drop('Geography',axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


#### Scaling the data

In [21]:
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

#### Predict Churn

In [22]:
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


array([[0.03936908]], dtype=float32)

In [23]:
prediction_proba=prediction[0][0]

In [16]:
prediction_proba

np.float32(0.03936908)

In [17]:
if prediction_proba>0.5:
    print('The Customer is likely to churn')
else:
    print('The Customer is not likely to churn')


The Customer is not likely to churn
